## 1. Import data from Factor Model

In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as pyplot

In [9]:
t_eval = np.load('data/t_eval.npy', allow_pickle = True)
returns = pd.read_csv('data/returns.csv', index_col = 0, parse_dates = True)

print(len(t_eval))
returns


83


,AAPL,ABBV,AEP,AMT,AMZN,APD,AVGO,BAC,BLK,CAT,...,SLB,SO,SPG,T,TSLA,UNH,UNP,WFC,WMT,XOM
Date,,,,,,,,,,,,,,,,,,,,,
2013-02-01,-0.031422,0.018084,0.032583,0.018862,-0.004644,-0.012545,-0.044299,-0.007982,0.014579,-0.063137,...,-0.002566,0.028546,-0.008275,0.048665,-0.074129,-0.032398,0.042081,0.014265,0.011796,-0.004679
2013-03-01,0.008662,0.099438,0.049106,-0.008801,0.008365,0.008994,0.048484,0.082105,0.069012,-0.060239,...,-0.034919,0.041559,0.005168,0.021488,0.084208,0.067982,0.043122,0.053017,0.055642,0.012602
2013-04-01,0.000271,0.121536,0.055981,0.087932,-0.048751,0.006270,-0.111507,0.010616,0.043753,-0.026802,...,-0.006161,0.027537,0.116044,0.020769,0.354112,0.050323,0.038231,0.026412,0.044325,-0.012507
2013-05-01,0.015574,-0.066551,-0.115501,-0.072696,0.058869,0.082199,0.165970,0.104060,0.046554,0.019661,...,-0.018989,-0.094067,-0.067542,-0.052416,0.593717,0.044067,0.044029,0.065476,-0.037761,0.016494
2013-06-01,-0.119299,-0.032134,-0.013318,-0.061870,0.031051,-0.030544,-0.009320,-0.060350,-0.083436,-0.039341,...,-0.018937,0.015880,-0.046109,0.011649,0.093672,0.044500,0.002182,0.025443,0.001293,0.005566
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-08-01,0.030684,0.066755,0.021775,0.016470,-0.046413,0.062176,0.013231,0.010856,0.028467,0.032259,...,-0.093262,0.033902,0.086778,0.048106,-0.080549,0.024076,0.037234,-0.014770,0.117912,-0.005496
2024-09-01,0.018473,0.005942,0.031888,0.037234,0.042931,0.065547,0.057752,-0.026609,0.051542,0.093803,...,-0.047484,0.051096,0.009930,0.100323,0.200441,-0.009397,-0.032942,-0.026840,0.047416,0.001895
2024-10-01,-0.030902,0.031845,-0.038246,-0.085324,0.000376,0.042060,-0.012669,0.059026,0.038557,-0.038893,...,-0.039327,0.009381,0.013132,0.024249,-0.046071,-0.031585,-0.060242,0.139092,0.014751,-0.003761


## 2. Rolling Covariance Matrix and PCA

In [ ]:
from sklearn.decomposition import PCA

lookback = 36
k = 5
Sigma = {}

for t in range(lookback, len(returns)):

    # Lookback window
    window = returns.iloc[t-lookback:t]

    # PCA deconstruction
    pca = PCA(n_components = k)
    pca.fit(window)

    B = pca.components_.T
    F = np.diag(pca.explained_variance_)

    Sigma_pca = B @ F @ B.T

    sample_cov = window.cov().values

    D = np.diag(np.diag(sample_cov - Sigma_pca))

    Sigma_t = Sigma_pca + D

    Sigma[returns.index[t]] = Sigma_t




## 3. Mean-Variance Optimisation

{Timestamp('2016-02-01 00:00:00'): array([[ 4.25223387e-03,  1.91049437e-03,  4.84269435e-05, ...,
          1.01789555e-03, -2.81013546e-04,  1.28368165e-03],
        [ 1.91049437e-03,  4.59421869e-03,  9.69252605e-04, ...,
          1.56523920e-03,  6.04990151e-04,  1.35263178e-03],
        [ 4.84269435e-05,  9.69252605e-04,  2.49679446e-03, ...,
          4.55950142e-05,  9.97025205e-04,  2.36284328e-04],
        ...,
        [ 1.01789555e-03,  1.56523920e-03,  4.55950142e-05, ...,
          1.37867114e-03,  1.95640390e-04,  7.49673457e-04],
        [-2.81013546e-04,  6.04990151e-04,  9.97025205e-04, ...,
          1.95640390e-04,  2.57194572e-03,  9.24180894e-05],
        [ 1.28368165e-03,  1.35263178e-03,  2.36284328e-04, ...,
          7.49673457e-04,  9.24180894e-05,  1.84783031e-03]],
       shape=(60, 60)),
 Timestamp('2016-03-01 00:00:00'): array([[ 4.20656264e-03,  1.91256118e-03,  4.99598010e-05, ...,
          9.79842489e-04, -2.60098884e-04,  1.24501358e-03],
        [ 1.